## Loading the dataset

In [8]:
import pandas as pd
import numpy as np
import numpy as np
from scipy.sparse import csr_matrix
import os

In [2]:

path = "dataset/Original/global/global-genre_network-2019.csv"
df = pd.read_csv(path, sep="\t")

print(df.head)


<bound method NDFrame.head of               source          target  weight   avg_streams
0              latin       reggaeton     479  1.060468e+08
1              latin           latin     282  1.048953e+08
2            pop rap             rap     222  5.385630e+07
3                rap            trap     218  6.656220e+07
4            hip hop             rap     203  5.405006e+07
...              ...             ...     ...           ...
2923   latin hip hop      latin rock       1  4.541218e+07
2924      latin rock  reggaeton flow       1  4.541218e+07
2925   latin hip hop     salsa choke       1  4.541218e+07
2926  reggaeton flow     salsa choke       1  4.541218e+07
2927         bachata  electro latino       1  1.053271e+08

[2928 rows x 4 columns]>


## Building a co-occurance matrix 

In [3]:
genres = pd.unique(df[["source", "target"]].values.ravel())
genre_to_idx = {g: i for i, g in enumerate(genres)}

n = len(genres)
adj_matrix = np.zeros((n, n), dtype=float)

for _, row in df.iterrows():
    i = genre_to_idx[row["source"]]
    j = genre_to_idx[row["target"]]
    w = row["weight"]

    adj_matrix[i, j] = w
    adj_matrix[j, i] = w

In [7]:
print("shape:", adj_matrix.shape)
print("max weight:", adj_matrix.max())
print("symmetric?", np.allclose(adj_matrix, adj_matrix.T))
print("nonzeros:", np.count_nonzero(adj_matrix))

shape: (310, 310)
max weight: 479.0
symmetric? True
nonzeros: 5782


### matrix check

In [4]:
print("Matrix shape:", adj_matrix.shape)
max_value = adj_matrix.max()
print("Max weight:", max_value)

Matrix shape: (310, 310)
Max weight: 479.0


### creating embeddings

In [5]:

A = adj_matrix.copy()

# (Recommended for embeddings) remove self-loops for training
# np.fill_diagonal(A, 0)

# (Recommended for first run) binarize
A_bin = (A > 0).astype(np.float32)

# Convert to sparse CSR (what many repos use internally)
genre_graph = csr_matrix(A_bin)

print("shape:", genre_graph.shape)
print("nonzeros:", genre_graph.nnz)
print("density:", genre_graph.nnz / (genre_graph.shape[0] * genre_graph.shape[1]))

shape: (310, 310)
nonzeros: 5782
density: 0.06016649323621228


In [6]:
A_bin = np.maximum(A_bin, A_bin.T)
genre_graph = csr_matrix(A_bin)